# VA-AFS Colab Runner

Colab 또는 VS Code 로컬 커널에서 실행한다. 로컬 커널에서는 `/content`를 쓰지 않고 현재 레포 아래에 결과를 저장한다.

In [1]:
try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except ImportError:
    IN_COLAB = False
    print('Local VS Code kernel: skip Google Drive mount.')

Mounted at /content/drive


Colab에서는 데이터 zip 4개를 아래 경로에 둔다. VS Code 로컬 커널에서는 이미 로컬에 풀린 프로젝트/데이터를 사용한다.

```text
/content/drive/MyDrive/AFS/data/
  all_sqe.zip
  nturgbd_skeletons_s001_to_s017.zip
  nturgbd_skeletons_s018_to_s032.zip
  videos.zip
```

In [2]:
from pathlib import Path

DATA_DIR = '/content/drive/MyDrive/AFS/data' if IN_COLAB else str((Path.cwd() / 'data').resolve())
print('DATA_DIR:', DATA_DIR)
!ls -lh "$DATA_DIR"

DATA_DIR: /content/drive/MyDrive/AFS/data
total 11G
-rw------- 1 root root  14M May  2 18:29 all_sqe.zip
-rw------- 1 root root 5.8G May  3 10:07 nturgbd_skeletons_s001_to_s017.zip
-rw------- 1 root root 4.5G May  2 21:05 nturgbd_skeletons_s018_to_s032.zip
-rw------- 1 root root 358M May  2 22:33 videos.zip


In [3]:
import os
import shutil
import subprocess
from pathlib import Path

if IN_COLAB:
    os.chdir('/content')
    shutil.rmtree('/content/src', ignore_errors=True)
    subprocess.run(['git', 'clone', '--branch', 'main', 'https://github.com/Kiim-Miin-Su/VA-AFS.git', '/content/src'], check=True)
    os.chdir('/content/src')
else:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'VA-AFS').exists() and (candidate / 'BlockGCN').exists():
            os.chdir(candidate)
            break
    print('Local VS Code kernel: using local repository.')

print('cwd:', Path.cwd())
subprocess.run(['git', 'log', '--oneline', '-1'], check=False)
print('setup_colab.py exists:', Path('setup_colab.py').exists())

cwd: /content/src
setup_colab.py exists: True


In [4]:
import subprocess

if IN_COLAB:
    subprocess.run(['python', 'setup_colab.py', '--data_dir', DATA_DIR, '--install', '--verify'], check=True)
else:
    print('Local VS Code kernel: skip Colab setup. Use the local environment and local data paths.')

GPU 확인.

In [5]:
import torch

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    DEVICE = '0'
    print('cuda device:', torch.cuda.get_device_name(0))
elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    DEVICE = 'mps'
    print('mps available: True')
else:
    DEVICE = 'cpu'
    print('GPU 런타임이 아니면 Colab 메뉴에서 런타임 > 런타임 유형 변경 > T4 GPU를 선택한다.')
print('selected device:', DEVICE)

torch: 2.10.0+cu128
cuda available: True
cuda device: Tesla T4
selected device: 0


빠른 동작 확인.

In [6]:
# !python -m pip install torch-topological

In [7]:
!python VA-AFS/run_colab_pipeline.py \
  --sample_size 200 \
  --num_epoch 2 \
  --batch_size 8 \
  --test_batch_size 8 \
  --num_worker 1 \
  --device "$DEVICE"


[cwd] /content/src/VA-AFS
[cmd] /usr/bin/python3 prepare_ntu_subset.py --sample_size 200 --test_ratio 0.2 --seed 1 --sampling_strategy balanced_fallback_random --output_dir /content/src/BlockGCN/data/ntu_subset_200 --force
Created NTU subset directory: /content/src/BlockGCN/data/ntu_subset_200
Selection mode: balanced sample_size=200, test_ratio=0.2, seed=1, requested_strategy=balanced_fallback_random
Selected samples: 200
CS train samples: 160
CS test samples: 40
First sample: S001C001P004R001A060

Next steps:
cd /content/src/BlockGCN/data/ntu_subset_200
python get_raw_skes_data.py
python get_raw_denoised_data.py
python seq_transformation.py

Expected output:
/content/src/BlockGCN/data/ntu_subset_200/NTU60_CS.npz

[cwd] /content/src/BlockGCN/data/ntu_subset_200
[cmd] /usr/bin/python3 get_raw_skes_data.py
Found 200 available skeleton files.
Reading data from S001C001P004R001A060.skeleton
Reading data from S001C001P007R002A007.skeleton
Reading data from S001C002P001R002A029.skeleton
Re

발표용 실행.

In [ ]:
!python VA-AFS/run_colab_pipeline.py \
  --sample_size 18000 \
  --num_epoch 80 \
  --batch_size 64 \
  --test_batch_size 64 \
  --num_worker 2 \
  --device "$DEVICE"

Streaming output truncated to the last 5000 lines.
Processing S014C001P015R002A009
Processing S014C001P015R002A010
Processing S014C001P015R002A011
Processing S014C001P015R002A016
Processing S014C001P015R002A017
Processing S014C001P015R002A019
Processing S014C001P015R002A021
Processing S014C001P015R002A023
Processing S014C001P015R002A024
Processing S014C001P015R002A025
Processing S014C001P015R002A028
Processing S014C001P015R002A030
Processing S014C001P015R002A034
Processing S014C001P015R002A035
Processing S014C001P015R002A036
Processing S014C001P015R002A037
Processing S014C001P015R002A039
Processing S014C001P015R002A041
Processing S014C001P015R002A042
Processing S014C001P015R002A043
Processing S014C001P015R002A044
Processing S014C001P015R002A047
Processing S014C001P015R002A048
Processing S014C001P015R002A052
Processing S014C001P015R002A053
Processing S014C001P015R002A054
Processing S014C001P015R002A056
Processing S014C001P015R002A057
Processing S014C001P017R001A004
Processing S014C001P0

전체 NTU60 데이터 실행. 시간이 오래 걸리므로 발표용 subset 실행 대신 필요할 때만 실행한다.

In [ ]:
# !python VA-AFS/run_colab_pipeline.py \
#   --full_data \
#   --num_epoch 80 \
#   --batch_size 64 \
#   --test_batch_size 64 \
#   --num_worker 2 \
#   --device "$DEVICE"

발표용 결과 시각화 함수.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

try:
    from IPython.display import Image, display
except ImportError:
    Image = None
    def display(value):
        print(value)

def _resolve_project_root():
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / 'VA-AFS').exists() and (candidate / 'BlockGCN').exists():
            return candidate
    colab_root = Path('/content/src')
    if colab_root.exists():
        return colab_root
    return cwd


ROOT = _resolve_project_root()
OUTPUT_DIR = ROOT / 'VA-AFS' / 'outputs'
PLOT_DIR = OUTPUT_DIR / 'presentation_plots'
print('Project root:', ROOT)
print('Presentation plot dir:', PLOT_DIR)


def _parse_run_name(path):
    name = Path(path).stem
    sample_match = re.search(r'subset_?(?P<sample>\d+)', name)
    epoch_match = re.search(r'_e(?P<epoch>\d+)$', name)
    tau_match = re.search(r'tau(?P<tau>[0-9.]+)', name)
    k_match = re.search(r'_k(?P<k>\d+)', name)
    w_match = re.search(r'_w(?P<w>\d+)', name)
    variant = 'vaafs' if 'vaafs' in name.lower() else 'original'
    is_full_data = 'ntu_full' in name.lower()

    return {
        'run': name,
        'sample_size': int(sample_match.group('sample')) if sample_match else None,
        'num_epoch': int(epoch_match.group('epoch')) if epoch_match else None,
        'variant': variant,
        'full_data': is_full_data,
        'tau': float(tau_match.group('tau')) if tau_match else None,
        'k_max': int(k_match.group('k')) if k_match else None,
        'window_size': int(w_match.group('w')) if w_match else None,
    }


def _normalize_accuracy(value):
    value = float(value)
    return value / 100.0 if value > 1.0 else value


def _read_accuracy(log_path):
    text = Path(log_path).read_text(errors='ignore')
    patterns = [
        r'Parsed Top-1 accuracy:\s*([0-9.]+)',
        r'Accuracy:\s*([0-9.]+)',
        r'Top1:\s*([0-9.]+)%',
        r'Top1 Acc:\s*([0-9.]+)%',
    ]
    matches = []
    for pattern in patterns:
        matches.extend(re.findall(pattern, text))
    if not matches:
        return None
    return _normalize_accuracy(matches[-1])


def collect_accuracy_results(acc_dir=None):
    acc_dir = Path(acc_dir) if acc_dir is not None else OUTPUT_DIR / 'blockgcn_acc'
    rows = []
    for log_path in sorted(Path(acc_dir).glob('**/log.txt')):
        accuracy = _read_accuracy(log_path)
        if accuracy is None:
            continue
        info = _parse_run_name(log_path.parent)
        rows.append({
            **info,
            'accuracy': accuracy,
            'accuracy_percent': accuracy * 100.0,
            'log_path': str(log_path),
        })
    return pd.DataFrame(rows)


def _valid_frame_counts(values):
    if values.ndim < 2:
        raise ValueError(f'Expected sequence array with at least 2 dimensions, got {values.shape}')
    feature_axes = tuple(range(2, values.ndim))
    return np.any(values != 0, axis=feature_axes).sum(axis=1).astype(np.int32)


def _candidate_original_npz_paths(info):
    if info.get('full_data'):
        return [ROOT / 'BlockGCN' / 'data' / 'ntu' / 'NTU60_CS.npz']
    sample_size = info.get('sample_size')
    if sample_size is None:
        return []
    return [ROOT / 'BlockGCN' / 'data' / f'ntu_subset_{sample_size}' / 'NTU60_CS.npz']


def _load_frame_counts(npz_path, split, info=None):
    npz_path = Path(npz_path)
    info = dict(info) if info is not None else _parse_run_name(npz_path)
    original_key = f'{split}_original_counts'
    selected_key = f'{split}_selected_counts'
    x_key = f'x_{split}'

    with np.load(npz_path) as data:
        if original_key in data and selected_key in data:
            return data[original_key], data[selected_key], 'metadata'
        if x_key not in data:
            return None, None, 'missing'
        selected_counts = _valid_frame_counts(data[x_key])

    for original_npz in _candidate_original_npz_paths(info):
        if not original_npz.exists():
            continue
        with np.load(original_npz) as original_data:
            if x_key not in original_data:
                continue
            original_counts = _valid_frame_counts(original_data[x_key])
        if len(original_counts) == len(selected_counts):
            return original_counts, selected_counts, 'inferred'

    return None, None, 'missing'


def collect_frame_ratio_results(npz_dir=None):
    npz_dir = Path(npz_dir) if npz_dir is not None else OUTPUT_DIR / 'blockgcn_npz'
    rows = []
    for npz_path in sorted(Path(npz_dir).glob('*.npz')):
        info = _parse_run_name(npz_path)
        for split in ('train', 'test'):
            original_counts, selected_counts, counts_source = _load_frame_counts(npz_path, split, info)
            if original_counts is None or selected_counts is None:
                continue

            valid = original_counts > 0
            total_original = int(original_counts[valid].sum())
            total_selected = int(selected_counts[valid].sum())
            processed_ratio = total_selected / max(total_original, 1)

            rows.append({
                **info,
                'split': split,
                'samples': int(valid.sum()),
                'original_frames': total_original,
                'selected_frames': total_selected,
                'processed_frame_ratio': processed_ratio,
                'frame_reduction_ratio': 1.0 - processed_ratio,
                'counts_source': counts_source,
                'npz_path': str(npz_path),
            })
    return pd.DataFrame(rows)


def _filter_latest(df, sample_size=None, num_epoch=None, split=None, variant=None, full_data=None):
    if df.empty:
        return df
    result = df.copy()
    if sample_size is not None and 'sample_size' in result:
        result = result[result['sample_size'] == sample_size]
    if num_epoch is not None and 'num_epoch' in result:
        result = result[result['num_epoch'] == num_epoch]
    if split is not None and 'split' in result:
        result = result[result['split'] == split]
    if variant is not None and 'variant' in result:
        result = result[result['variant'] == variant]
    if full_data is not None and 'full_data' in result:
        result = result[result['full_data'] == full_data]
    return result


def _save_and_show(fig, save_path):
    save_path = Path(save_path)
    save_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(save_path, dpi=200, bbox_inches='tight')
    print(f'Saved plot: {save_path}')
    plt.show()
    if Image is not None:
        display(Image(filename=str(save_path)))


def plot_frame_ratio(sample_size=20000, split='test', full_data=False, save_path=None):
    summary = _filter_latest(
        collect_frame_ratio_results(),
        sample_size=None if full_data else sample_size,
        split=split,
        variant='vaafs',
        full_data=full_data,
    )
    if summary.empty:
        npz_dir = OUTPUT_DIR / 'blockgcn_npz'
        available = ', '.join(p.name for p in sorted(npz_dir.glob('*.npz'))) or 'none'
        raise FileNotFoundError(
            f'No VA-AFS frame ratio result found for sample={"full" if full_data else sample_size}, '
            f'split={split}. Searched: {npz_dir}. Available npz files: {available}. '
            'Run the presentation pipeline cell, or rerun it with --force_vaafs if the npz was created by an older notebook.'
        )

    row = summary.sort_values('npz_path').iloc[-1]
    original_counts, selected_counts, _ = _load_frame_counts(row['npz_path'], split, row)
    if original_counts is None or selected_counts is None:
        raise FileNotFoundError(f'Could not load frame counts from {row["npz_path"]}')

    valid = original_counts > 0
    ratios = selected_counts[valid] / original_counts[valid]
    processed_ratio = float(row['processed_frame_ratio'])
    reduction_ratio = float(row['frame_reduction_ratio'])

    fig, (ax_bar, ax_hist) = plt.subplots(1, 2, figsize=(13, 4.8))
    bars = ax_bar.bar(
        ['Processed', 'Skipped'],
        [processed_ratio, reduction_ratio],
        color=['#2E7D32', '#C62828'],
    )
    ax_bar.set_ylim(0, 1)
    ax_bar.set_ylabel('Frame ratio')
    ax_bar.set_title('Total Frame Ratio')
    ax_bar.grid(axis='y', alpha=0.25)
    for bar in bars:
        value = bar.get_height()
        ax_bar.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.025,
            f'{value * 100:.1f}%',
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold',
        )

    ax_hist.hist(ratios, bins=np.linspace(0, 1, 11), color='#1565C0', edgecolor='white')
    ax_hist.axvline(processed_ratio, color='#C62828', linestyle='--', label=f'Mean {processed_ratio * 100:.1f}%')
    ax_hist.set_xlim(0, 1)
    ax_hist.set_xlabel('Processed ratio per sample')
    ax_hist.set_ylabel('Sample count')
    ax_hist.set_title('Per-sample Ratio Distribution')
    ax_hist.legend()
    ax_hist.grid(axis='y', alpha=0.25)

    fig.suptitle(
        f'VA-AFS Frame Selection | sample={"full" if full_data else sample_size}, split={split}, '
        f'tau={row["tau"]}, k={row["k_max"]}, window={row["window_size"]}',
        fontsize=13,
        fontweight='bold',
    )
    fig.tight_layout()
    if save_path is None:
        run_label = 'full' if full_data else f'subset{sample_size}'
        save_path = PLOT_DIR / f'{run_label}_{split}_frame_ratio.png'
    _save_and_show(fig, save_path)
    return summary


def plot_accuracy(sample_size=20000, num_epoch=80, full_data=False, save_path=None):
    df = _filter_latest(
        collect_accuracy_results(),
        sample_size=None if full_data else sample_size,
        num_epoch=num_epoch,
        full_data=full_data,
    )
    if df.empty:
        raise FileNotFoundError('Accuracy logs were not found. Run the pipeline first.')

    order = ['original', 'vaafs']
    df = df.sort_values('variant').drop_duplicates('variant', keep='last')
    df['variant'] = pd.Categorical(df['variant'], categories=order, ordered=True)
    df = df.sort_values('variant')

    fig, ax = plt.subplots(figsize=(7.2, 4.8))
    colors = ['#455A64' if variant == 'original' else '#2E7D32' for variant in df['variant'].astype(str)]
    bars = ax.bar(df['variant'].astype(str), df['accuracy_percent'], color=colors)
    ax.set_ylim(0, max(100, float(df['accuracy_percent'].max()) + 5))
    ax.set_ylabel('Top-1 accuracy (%)')
    ax.set_title(f'BlockGCN Accuracy Comparison | sample={"full" if full_data else sample_size}, epoch={num_epoch}')
    ax.grid(axis='y', alpha=0.25)
    for bar in bars:
        value = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            value + 0.8,
            f'{value:.2f}%',
            ha='center',
            va='bottom',
            fontsize=11,
            fontweight='bold',
        )

    if {'original', 'vaafs'}.issubset(set(df['variant'].astype(str))):
        original = float(df.loc[df['variant'].astype(str) == 'original', 'accuracy_percent'].iloc[-1])
        vaafs = float(df.loc[df['variant'].astype(str) == 'vaafs', 'accuracy_percent'].iloc[-1])
        ax.text(
            0.5,
            0.95,
            f'Delta: {vaafs - original:+.2f} pp',
            transform=ax.transAxes,
            ha='center',
            va='top',
            bbox={'facecolor': 'white', 'alpha': 0.85, 'edgecolor': '#B0BEC5'},
        )

    fig.tight_layout()
    if save_path is None:
        run_label = 'full' if full_data else f'subset{sample_size}'
        save_path = PLOT_DIR / f'{run_label}_e{num_epoch}_accuracy.png'
    _save_and_show(fig, save_path)
    return df


def show_threshold_selection_plots(limit=3, plot_dir=None):
    plot_dir = Path(plot_dir) if plot_dir is not None else OUTPUT_DIR / 'threshold' / 'plots'
    plots = sorted(Path(plot_dir).glob('*_selection.png'))[-limit:]
    if not plots:
        raise FileNotFoundError('Saved threshold selection plots were not found.')
    if Image is None:
        return plots
    for plot_path in plots:
        print(plot_path)
        display(Image(filename=str(plot_path)))
    return plots


def show_presentation_figures(sample_size=20000, num_epoch=80, split='test', full_data=False):
    frame_summary = plot_frame_ratio(sample_size=sample_size, split=split, full_data=full_data)
    acc_summary = plot_accuracy(sample_size=sample_size, num_epoch=num_epoch, full_data=full_data)
    display(frame_summary)
    display(acc_summary)
    return frame_summary, acc_summary

발표용 실행 결과를 바로 확인한다.

In [ ]:
frame_summary, acc_summary = show_presentation_figures(sample_size=18000, num_epoch=80, split='test')

전체 NTU60 실행 결과를 시각화할 때 사용한다.

In [ ]:
# full_frame_summary, full_acc_summary = show_presentation_figures(num_epoch=80, split='test', full_data=True)